<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/OtomatikFeatureSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn
import matplotlib

In [ ]:
df = pd.read_csv('/content/Feature Selection.csv', encoding='latin1', sep=';', engine='python')
display(df.head())

In [ ]:
df.info()

In [ ]:
df['opened_at'] = pd.to_datetime(df['opened_at'], errors='coerce')
df['resolved_at'] = pd.to_datetime(df['resolved_at'], errors='coerce')
display(df.head())

In [ ]:
df.info()

### Veri Seti Boyutu

In [ ]:
print(f"DataFrame boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")

### Kategorik Veri Analizi

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    print(f"\n--- Sütun: {col} ---")
    print(df[col].value_counts())
    print(f"Eşsiz değer sayısı: {df[col].nunique()}")

### Sayısal Veri Analizi

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    print(f"\n--- Sütun: {col} ---")
    display(df[col].describe())

In [ ]:
df = df.drop(columns=['escalation', 'due_date'])
display(df.head())

In [ ]:
df.info()

### Eksik Değer Analizi

In [ ]:
missing_values = df.isnull().sum()
display(missing_values[missing_values > 0])

### Gürültülü Veri (Aykırı Değer) Tespiti

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    if df[col].isnull().any():
        print(f"Sütun '{col}' eksik değerler içerdiği için aykırı değer tespiti atlanıyor.")
        continue

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    if not outliers.empty:
        print(f"\n--- Sütun '{col}' için Aykırı Değerler ---")
        print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
        print(f"Alt Sınır: {lower_bound}, Üst Sınır: {upper_bound}")
        print(f"Tespit Edilen Aykırı Değer Sayısı: {len(outliers)}")
        display(outliers.head())
    else:
        print(f"\n--- Sütun '{col}' için Aykırı Değer Bulunamadı ---")

### Aykırı Değerlerin Kutu Grafiği ile Görselleştirilmesi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(y=df['calendar_duration'])
plt.title('Calendar Duration Sütunu Aykırı Değerler Kutusu Grafiği')
plt.ylabel('Calendar Duration')
plt.show()

### Sayısal ve Kategorik Değişken Analizi

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64', 'datetime64[ns]']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print(f"Sayısal değişken sayısı: {len(numerical_cols)}")
print(f"Kategorik değişken sayısı: {len(categorical_cols)}")

### Her Sütunun Veri Tipi Bilgisi

In [ ]:
df.info()

### Sayısal Sütunların İstatistiksel Özellikleri

In [ ]:
display(df.describe())

### Kategorik Sütunların İstatistiksel Özellikleri

In [ ]:
display(df.describe(include='object'))

### Korelasyon Matrisi

In [ ]:
numerical_cols_for_corr = df.select_dtypes(include=['int64', 'float64']).columns
correlation_matrix = df[numerical_cols_for_corr].corr()
display(correlation_matrix)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Sayısal Sütunlar Arası Korelasyon Matrisi Isı Haritası')
plt.show()

### Kategorik Değişkenler Arası İlişki Analizi: 'category' ve 'impact'

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1))) if min((kcorr-1), (rcorr-1)) > 0 else 0

# 'category' ve 'impact' sütunları
contingency_table = pd.crosstab(df['category'], df['impact'])
display(contingency_table)

# Cramer's V değeri
cv = cramers_v(df['category'], df['impact'])
print(f"'category' ve 'impact' arasındaki Cramer's V değeri: {cv:.2f}")

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(contingency_table, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Category ve Impact Arası İlişki Isı Haritası')
plt.xlabel('Impact')
plt.ylabel('Category')
plt.show()

### Veri Temizliği ve Ön İşleme

In [ ]:
df_processed = df.dropna().copy()
print(f"Eksik değerler düşürüldükten sonra DataFrame boyutu: {df_processed.shape[0]} satır, {df_processed.shape[1]} sütun")

display(df_processed.isnull().sum()[df_processed.isnull().sum() > 0])

### Yüksek Kardinaliteli Sütunların Ele Alınması: 'subcategory' Sütununun Kaldırılması

In [ ]:
df_processed = df_processed.drop(columns=['subcategory'])
print("'subcategory' sütunu kaldırıldı.")
display(df_processed.head())

### Kategorik Sütunlara One-Hot Encoding Uygulanması

In [ ]:
categorical_cols_to_encode = ['category', 'assignment_group', 'close_code', 'impact']
df_processed = pd.get_dummies(df_processed, columns=categorical_cols_to_encode, prefix=categorical_cols_to_encode, dtype=int)

print("One-Hot Encoding uygulandı.")
display(df_processed.head())
print(f"Encoding sonrası DataFrame sütun sayısı: {df_processed.shape[1]}")

### Sayısal Sütunların Ölçeklendirilmesi (StandardScaler)

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols_to_scale = ['priority', 'calendar_duration']

scaler = StandardScaler()

df_processed[numerical_cols_to_scale] = scaler.fit_transform(df_processed[numerical_cols_to_scale])

print("Sayısal sütunlar ölçeklendirildi.")
display(df_processed.head())

### Makine Öğrenimi İçin Hazırlanmış Veri Setinin Son Durumu

In [ ]:
df_processed.info()

In [ ]:
from sklearn.model_selection import train_test_split
X = df_processed.drop(columns=['priority', 'opened_at', 'resolved_at'])
y = df_processed['priority']

print("Özellikler (X) ve hedef değişken (y) ayrıldı.")
display(X.head())
display(y.head())

### Veri Setinin Train ve Test Olarak Bölünmesi

In [ ]:
# Veri setini eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train boyutu: {X_train.shape}")
print(f"X_test boyutu: {X_test.shape}")
print(f"y_train boyutu: {y_train.shape}")
print(f"y_test boyutu: {y_test.shape}")

### Model Kurulumu ve Eğitimi: Random Forest Regressor (Ham Veri)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

random_forest_reg_model = RandomForestRegressor(n_estimators=100, random_state=42)

random_forest_reg_model.fit(X_train, y_train)

print("Random Forest Regressor modeli başarıyla eğitildi.")

### Random Forest Regressor Model Değerlendirmesi

In [ ]:
y_pred_rf = random_forest_reg_model.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Mean Squared Error (MSE): {mse_rf:.4f}")
print(f"Random Forest R-squared (R2 Score): {r2_rf:.4f}")

results_rf_df = pd.DataFrame({'Gerçek Değer': y_test, 'Tahmin Edilen Değer': y_pred_rf})
display(results_rf_df.head())

### Random Forest Regressor: Gerçek ve Tahmin Edilen Değerlerin Görselleştirilmesi

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred_rf, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2) # y=x doğrusu
plt.title('Random Forest: Gerçek ve Tahmin Edilen Değerler Karşılaştırması')
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.grid(True)
plt.show()

### Logistic Regression Model Kurulumu ve Eğitimi (Ham Veri)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_classification = df.loc[df_processed.index, 'priority'].astype(int)

X_classification = X.copy()

print("Özellikler (X_classification) ve hedef değişken (y_classification) ayrıldı.")
display(X_classification.head())
display(y_classification.head())

# Veri setini eğitim ve test setlerine ayırma
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_classification, y_classification, test_size=0.2, random_state=42, stratify=y_classification)

print(f"X_train_clf boyutu: {X_train_clf.shape}")
print(f"X_test_clf boyutu: {X_test_clf.shape}")
print(f"y_train_clf boyutu: {y_train_clf.shape}")
print(f"y_test_clf boyutu: {y_test_clf.shape}")

log_reg_model = LogisticRegression(multi_class='ovr', solver='liblinear', random_state=42, max_iter=1000)

# Modeli eğitim verisiyle eğit
log_reg_model.fit(X_train_clf, y_train_clf)

print("Ham veri üzerinde Logistic Regression modeli başarıyla eğitildi.")

### Logistic Regression Model Değerlendirmesi

In [ ]:
# Test seti üzerinde tahminler yap
y_pred_clf = log_reg_model.predict(X_test_clf)
y_proba_clf = log_reg_model.predict_proba(X_test_clf)

# Sınıflandırma metriklerini hesapla
accuracy = accuracy_score(y_test_clf, y_pred_clf)
precision_macro = precision_score(y_test_clf, y_pred_clf, average='macro', zero_division=0)
recall_macro = recall_score(y_test_clf, y_pred_clf, average='macro', zero_division=0)
f1_macro = f1_score(y_test_clf, y_pred_clf, average='macro', zero_division=0)

# ROC-AUC (Çok sınıflı için 'ovr' stratejisi)
roc_auc_ovr = roc_auc_score(y_test_clf, y_proba_clf, multi_class='ovr')

metrics_data = {
    'Metric': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)', 'ROC-AUC (OvR)'],
    'Value': [accuracy, precision_macro, recall_macro, f1_macro, roc_auc_ovr]
}
metrics_df = pd.DataFrame(metrics_data)

display(metrics_df)

print("Sınıflandırma metrikleri başarıyla hesaplandı.")

### Confusion Matrix ile Logistic Regression Model Performansının Görselleştirilmesi

In [ ]:
# Confusion matrix'i hesapla
cm = confusion_matrix(y_test_clf, y_pred_clf)

# Confusion matrix'i görselleştir
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=log_reg_model.classes_,
            yticklabels=log_reg_model.classes_)
plt.title('Ham Veri - Logistic Regression Confusion Matrix')
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.show()

### SHAP (SHapley Additive exPlanations) ile Model Yorumlanması

In [ ]:
!pip install shap

import shap
import matplotlib.pyplot as plt

print("SHAP explainer kuruluyor...")

# LinearExplainer kullanıyoruz çünkü LogisticRegression doğrusal bir modeldir.
explainer = shap.LinearExplainer(log_reg_model, X_train_clf)

# Test seti üzerindeki SHAP değerlerini hesapla
shap_values = explainer.shap_values(X_test_clf)

print("SHAP değerleri hesaplandı.")

### SHAP Bar Plot (Ortalama Etki Büyüklükleri)

In [ ]:

if isinstance(shap_values, list):
    # Çoklu sınıflandırma için tüm sınıfların ortalama mutlak SHAP değerlerini alalım
    abs_shap_values = np.abs(np.array(shap_values)).mean(axis=0)
    shap.bar_plot(abs_shap_values, X_test_clf.columns)
else:
    shap.bar_plot(np.abs(shap_values).mean(axis=(0, 2)), X_test_clf.columns)

plt.tight_layout()
plt.show()

### Random Forest Regressor için SHAP Analizi

In [ ]:
import shap
import matplotlib.pyplot as plt

print("Random Forest Regressor için SHAP explainer kuruluyor...")

# Random Forest gibi ağaç tabanlı modeller için TreeExplainer kullanılır.
explainer_rf = shap.TreeExplainer(random_forest_reg_model)

# Test seti üzerindeki SHAP değerlerini hesapla
#shap_values_rf = explainer_rf.shap_values(X_test)

print("Random Forest Regressor SHAP değerleri hesaplandı.")

### Random Forest Regressor SHAP (Özet Grafiği)

In [ ]:
shap.summary_plot(shap_values_rf, X_test, feature_names=X_test.columns)